**Extracting absolute counts for Representative Days**

After identifying the representative days, the original absolute detector counts were extracted for each selected scenario. This step was necessary because the K-means clustering was based on normalized daily profiles only for pattern recognition, whereas the subsequent demand reconstruction requires absolute traffic volumes.

For each representative day, the detector data were filtered to the corresponding scenario period and aggregated by detector. In addition, the full 15-minute detector time series of the selected day was preserved for later calibration and validation. These aggregated absolute counts form the basis for the subsequent mapping of detector observations to SUMO network edges and for the generation of demand inputs for route-based traffic reconstruction.

In [1]:
import pandas as pd
from pathlib import Path

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau"
)

IN_FILE = BASE_DIR / "rebuilt_common_days_routesampler_base" / "all_intersections_routesampler_base_long.csv"

OUT_DIR = BASE_DIR / "representative_day_routesampler_base"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [
    {
        "name": "weekday_morning",
        "date": "2025-06-17",
        "start_time": "08:00:00",
        "end_time": "09:00:00"
    },
    {
        "name": "weekday_evening",
        "date": "2025-06-19",
        "start_time": "16:00:00",
        "end_time": "17:00:00"
    },
    {
        "name": "weekend_morning",
        "date": "2025-06-21",
        "start_time": "08:00:00",
        "end_time": "09:00:00"
    },
    {
        "name": "weekend_evening",
        "date": "2025-06-07",
        "start_time": "16:00:00",
        "end_time": "17:00:00"
    },
    {
        "name": "holiday_morning",
        "date": "2025-06-09",
        "start_time": "08:00:00",
        "end_time": "09:00:00"
    },
    {
        "name": "holiday_evening",
        "date": "2025-06-09",
        "start_time": "16:00:00",
        "end_time": "17:00:00"
    }
]

# ============================================================
# LOAD
# ============================================================

df = pd.read_csv(IN_FILE)

df["timestamp_local"] = pd.to_datetime(df["timestamp_local"], errors="coerce")
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce", utc=True)
df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")

if "time" not in df.columns:
    df["time"] = df["timestamp_local"].dt.strftime("%H:%M:%S")

df["count"] = pd.to_numeric(df["count"], errors="coerce").fillna(0)
df["occupancy"] = pd.to_numeric(df["occupancy"], errors="coerce").fillna(0)
df["velocity"] = pd.to_numeric(df["velocity"], errors="coerce").fillna(0)
df["begin_s"] = pd.to_numeric(df["begin_s"], errors="coerce")
df["end_s"] = pd.to_numeric(df["end_s"], errors="coerce")

# ============================================================
# EXTRACT PER SCENARIO
# ============================================================

summary_rows = []

for sc in SCENARIOS:
    name = sc["name"]
    rep_day = sc["date"]
    start_time = sc["start_time"]
    end_time = sc["end_time"]

    temp = df[
        (df["date"] == rep_day) &
        (df["time"] >= start_time) &
        (df["time"] < end_time)
    ].copy()

    temp = temp.sort_values(
        ["intersection", "timestamp_local", "detector_id"]
    ).reset_index(drop=True)

    # 1) raw long representative-day base
    keep_cols = [
        "intersection",
        "date",
        "timestamp_utc",
        "timestamp_local",
        "time",
        "begin_s",
        "end_s",
        "detector_id",
        "edge_id",
        "movement",
        "count_type",
        "from_edge",
        "to_edge",
        "count",
        "occupancy",
        "velocity",
    ]

    long_out = OUT_DIR / f"{name}_routesampler_base_long.csv"
    temp[keep_cols].to_csv(long_out, index=False, encoding="utf-8-sig")

    # 2) 5-min detector intervals aggregated in SUMO-friendly format
    detector_interval = (
        temp.groupby(
            [
                "intersection", "date", "begin_s", "end_s",
                "detector_id", "edge_id", "count_type",
                "from_edge", "to_edge", "movement"
            ],
            dropna=False,
            as_index=False
        )
        .agg(
            count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_velocity=("velocity", "mean")
        )
        .sort_values(["begin_s", "intersection", "detector_id"])
    )

    detector_interval_out = OUT_DIR / f"{name}_detector_interval_counts.csv"
    detector_interval.to_csv(detector_interval_out, index=False, encoding="utf-8-sig")

    # 3) 15-min detector matrix
    detector_15 = (
        temp.groupby(
            [
                "intersection", "date", "detector_id", "edge_id",
                "count_type", "from_edge", "to_edge", "movement",
                "begin_s", "end_s"
            ],
            dropna=False,
            as_index=False
        )
        .agg(
            count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_velocity=("velocity", "mean")
        )
        .sort_values(["intersection", "detector_id", "begin_s"])
    )

    detector_15_out = OUT_DIR / f"{name}_detector_15min_counts.csv"
    detector_15.to_csv(detector_15_out, index=False, encoding="utf-8-sig")

    # 4) wide matrix only for quick inspection
    pivot = temp.pivot_table(
        index=["intersection", "detector_id", "edge_id", "count_type", "from_edge", "to_edge", "movement"],
        columns="time",
        values="count",
        aggfunc="sum",
        fill_value=0
    ).reset_index()

    pivot_out = OUT_DIR / f"{name}_detector_time_matrix.csv"
    pivot.to_csv(pivot_out, index=False, encoding="utf-8-sig")

    # 5) technical summary by detector
    detector_summary = (
        temp.groupby(
            ["intersection", "detector_id", "edge_id", "count_type", "from_edge", "to_edge", "movement"],
            dropna=False,
            as_index=False
        )
        .agg(
            total_count=("count", "sum"),
            mean_occupancy=("occupancy", "mean"),
            mean_velocity=("velocity", "mean"),
            n_intervals=("count", "size")
        )
        .sort_values(["intersection", "detector_id"])
    )

    detector_summary_out = OUT_DIR / f"{name}_detector_summary.csv"
    detector_summary.to_csv(detector_summary_out, index=False, encoding="utf-8-sig")

    summary_rows.append({
        "scenario": name,
        "date": rep_day,
        "start_time": start_time,
        "end_time": end_time,
        "n_rows": len(temp),
        "n_detectors": temp[["intersection", "detector_id"]].drop_duplicates().shape[0],
        "total_count": temp["count"].sum()
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(
    OUT_DIR / "representative_day_routesampler_base_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n============================================================")
print("REPRESENTATIVE DAY EXTRACTION SUMMARY")
print("============================================================")
print(summary_df.to_string(index=False))
print(f"\nSaved outputs to:\n{OUT_DIR}")

C:\Users\mogul\AppData\Local\Temp\ipykernel_21244\3769246055.py:60: DtypeWarning: Columns (4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(IN_FILE)



REPRESENTATIVE DAY EXTRACTION SUMMARY
       scenario       date start_time end_time  n_rows  n_detectors  total_count
weekday_morning 2025-06-17   08:00:00 09:00:00     140           35         8439
weekday_evening 2025-06-19   16:00:00 17:00:00     140           35         4452
weekend_morning 2025-06-21   08:00:00 09:00:00     140           35         4366
weekend_evening 2025-06-07   16:00:00 17:00:00     140           35         7490
holiday_morning 2025-06-09   08:00:00 09:00:00     140           35         1811
holiday_evening 2025-06-09   16:00:00 17:00:00     140           35         5976

Saved outputs to:
C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\representative_day_routesampler_base


***detector → edge → counts.xml***

In [27]:
import pandas as pd
import xml.etree.ElementTree as ET
from pathlib import Path
from xml.dom import minidom

# ============================================================
# SETTINGS
# ============================================================

BASE_DIR = Path(
    r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau"
)

IN_DIR = BASE_DIR / "representative_day_routesampler_base"
OUT_DIR = BASE_DIR / "routesampler_input"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SCENARIOS = [
    {
        "name": "weekday_morning",
        "input_csv": IN_DIR / "weekday_morning_detector_summary.csv",
        "begin": 0.0,
        "end": 3600.0
    },
    {
        "name": "weekday_evening",
        "input_csv": IN_DIR / "weekday_evening_detector_summary.csv",
        "begin": 0.0,
        "end": 3600.0
    },
    {
        "name": "weekend_morning",
        "input_csv": IN_DIR / "weekend_morning_detector_summary.csv",
        "begin": 0.0,
        "end": 3600.0
    },
    {
        "name": "weekend_evening",
        "input_csv": IN_DIR / "weekend_evening_detector_summary.csv",
        "begin": 0.0,
        "end": 3600.0
    },
    {
        "name": "holiday_morning",
        "input_csv": IN_DIR / "holiday_morning_detector_summary.csv",
        "begin": 0.0,
        "end": 3600.0
    },
    {
        "name": "holiday_evening",
        "input_csv": IN_DIR / "holiday_evening_detector_summary.csv",
        "begin": 0.0,
        "end": 3600.0
    },
]

# ============================================================
# EDGE COUNT PART
# ============================================================

EDGE_RULES = [
    # ---------------- LSA16 ----------------
    {"name": "LSA16_to_LSA10", "members": [("LD-LSA16", 13)], "edge": "-202157700#5"},
    {"name": "LSA16_W_exit", "members": [("LD-LSA16", 15)], "edge": "26202222#0"},
    {"name": "LSA16_S_approach", "members": [("LD-LSA16", 1), ("LD-LSA16", 2), ("LD-LSA16", 33)], "edge": "-202157703"},
   {"name": "LSA16_N_approach", "members": [("LD-LSA16", 6), ("LD-LSA16", 7)], "edge": "-1182936853"},
   {"name": "LSA16_N_approach", "members": [("LD-LSA16", 3), ("LD-LSA16", 4)], "edge": "-26202222#0.41"},
    {"name": "LSA16_E_approach", "members": [("LD-LSA16", 10), ("LD-LSA16", 11)], "edge": "202157700#3"},

    # ---------------- LSA10 ----------------
    {"name": "LSA10_S_approach", "members": [("LD-LSA10", 31)], "edge": "-1183449794"},
    {"name": "LSA10_E_approach", "members": [("LD-LSA10", 3), ("LD-LSA10", 4)], "edge": "75642281"},
    {"name": "LSA10_NW_approach", "members": [("LD-LSA10", 2)], "edge": "-13831317#2"},
    {"name": "LSA10_NE_approach", "members": [("LD-LSA10", 29)], "edge": "-25563346#2"},

    # ---------------- LSA1 ----------------
   {"name": "LSA1_W_approach", "members": [("LD-LSA1", 7), ("LD-LSA1", 9)], "edge": "-202234344"},
   {"name": "LSA1_E_approach", "members": [("LD-LSA1", 24), ("LD-LSA1", 26), ("LD-LSA1", 14)], "edge": "-237921040"},
   {"name": "LSA1_N_approach", "members": [("LD-LSA1", 40), ("LD-LSA1", 28), ("LD-LSA1", 30)], "edge": "237921036.41"},
]

# ============================================================
# TURN COUNT PART
# ============================================================

TURN_RULES = [
    # ---------------- LSA16 ----------------
    {"detector": ("LD-LSA16", 4), "from": "-26202222#0.41", "to": ["893429999#1"]},
    {"detector": ("LD-LSA16", 3), "from": "-26202222#0.41", "to": ["-202157700#5"]},
    {"detector": ("LD-LSA16", 1), "from": "-202157703", "to": ["-202157700#5"]},
    {"detector": ("LD-LSA16", 2), "from": "-202157703", "to": ["893429999#1"]},
    {"detector": ("LD-LSA16", 33), "from": "-202157703", "to": ["893429999#1"]},
    {"detector": ("LD-LSA16", 6), "from": "-1182936853", "to": ["26202222#0"]},
    {"detector": ("LD-LSA16", 7), "from": "-1182936853", "to": ["12528053#0"]},
    {"detector": ("LD-LSA16", 11), "from": "202157700#3", "to": ["12528053#0", "26202222#0"]},
    {"detector": ("LD-LSA16", 10), "from": "202157700#3", "to": ["893429999#1", "26202222#0"]},

    # ---------------- LSA10 ----------------
    {"detector": ("LD-LSA10", 27), "from": "-13831317#2.34", "to": ["25563346#0", "-84620353#2"]},
    {"detector": ("LD-LSA10", 26), "from": "-13831317#2.34", "to": ["17707991#0", "14811377"]},
    {"detector": ("LD-LSA10", 29), "from": "-25563346#2", "to": ["13831317#0"]},

    # ---------------- LSA1 ----------------
    {"detector": ("LD-LSA1", 7),  "from": "-202234344", "to": ["-237921036"]},
    {"detector": ("LD-LSA1", 9),  "from": "-202234344", "to": ["59617619", "-281402235#1"]},
    {"detector": ("LD-LSA1", 27), "from": "-59617619", "to": ["-237921036"]},
    {"detector": ("LD-LSA1", 13), "from": "-59617619", "to": ["12695647"]},
    {"detector": ("LD-LSA1", 23), "from": "-59617619", "to": ["12695647"]},

    # only THIS line has via
    {"detector": ("LD-LSA1", 26), "from": "-237921040", "to": ["-281402232"], "via": {"-281402232": "-59617619 -281402235#1"}},

    # north
    {"detector": ("LD-LSA1", 40), "from": "237921036.41", "to": ["12695647"]},
    {"detector": ("LD-LSA1", 28), "from": "237921036.41", "to": ["-281402235#1"]},
    {"detector": ("LD-LSA1", 30), "from": "237921036.41", "to": ["59617619"]},
]

# ============================================================
# STARTING SPLIT RATIOS
# ============================================================

TURN_SPLIT_RATIOS = {
    ("LD-LSA16", 10): [0.1, 0.9],
    ("LD-LSA16", 11): [0.7, 0.3],
    ("LD-LSA10", 27): [0.05, 0.95],
    ("LD-LSA10", 26): [0.1, 0.9],
    ("LD-LSA1", 9):   [0.8, 0.2],
}

# ============================================================
# HELPERS
# ============================================================

def prettify(elem):
    rough = ET.tostring(elem, encoding="utf-8")
    reparsed = minidom.parseString(rough)
    return reparsed.toprettyxml(indent="  ")

def split_count(total_count, n_parts, ratios=None):
    total_count = float(total_count)

    if ratios is None:
        ratios = [1.0 / n_parts] * n_parts

    if len(ratios) != n_parts:
        raise ValueError(f"Ratio count mismatch: {len(ratios)} vs {n_parts}")

    ratio_sum = sum(ratios)
    ratios = [r / ratio_sum for r in ratios]

    raw = [total_count * r for r in ratios]
    ints = [int(x) for x in raw]
    remainder = int(round(total_count - sum(ints)))

    frac_order = sorted(
        range(n_parts),
        key=lambda i: raw[i] - ints[i],
        reverse=True
    )

    for i in range(remainder):
        ints[frac_order[i % n_parts]] += 1

    return ints

def get_detector_total(df, detector_key):
    temp = df[(df["intersection"] == detector_key[0]) & (df["detector_id"] == detector_key[1])]
    if temp.empty:
        return 0.0
    return float(temp["total_count"].sum())

# ============================================================
# MAIN
# ============================================================

for sc in SCENARIOS:
    print("\n" + "=" * 70)
    print(f"SCENARIO: {sc['name']}")
    print("=" * 70)

    if not sc["input_csv"].exists():
        print(f"Missing input: {sc['input_csv']}")
        continue

    df = pd.read_csv(sc["input_csv"])
    df["detector_id"] = pd.to_numeric(df["detector_id"], errors="coerce").astype("Int64")
    df["total_count"] = pd.to_numeric(df["total_count"], errors="coerce").fillna(0)

    # ---------------- EDGE DATA ----------------
    edge_rows = []
    for rule in EDGE_RULES:
        total = 0.0
        for det_key in rule["members"]:
            total += get_detector_total(df, det_key)

        edge_rows.append({
            "edge": rule["edge"],
            "count": int(round(total)),
            "rule_name": rule["name"]
        })

    edge_counts = pd.DataFrame(edge_rows)
    edge_counts = (
        edge_counts.groupby("edge", as_index=False)["count"]
        .sum()
        .sort_values("edge")
    )

    print("\nEDGE COUNTS")
    print(edge_counts.to_string(index=False))

    edge_root = ET.Element("data")
    edge_interval = ET.SubElement(edge_root, "interval", {
        "id": sc["name"],
        "begin": str(sc["begin"]),
        "end": str(sc["end"])
    })

    for _, r in edge_counts.iterrows():
        ET.SubElement(edge_interval, "edge", {
            "id": str(r["edge"]),
            "entered": str(int(r["count"]))
        })

    edge_xml_path = OUT_DIR / f"edgeData_{sc['name']}.xml"
    edge_csv_path = OUT_DIR / f"edgeData_{sc['name']}.csv"

    edge_counts.to_csv(edge_csv_path, index=False, encoding="utf-8-sig")
    with open(edge_xml_path, "w", encoding="utf-8") as f:
        f.write(prettify(edge_root))

    print("Saved:", edge_csv_path)
    print("Saved:", edge_xml_path)

    # ---------------- TURN DATA ----------------
    turn_rows = []

    for rule in TURN_RULES:
        det_key = rule["detector"]
        total = get_detector_total(df, det_key)

        if total <= 0:
            continue

        from_edge = rule["from"]
        to_edges = rule["to"]
        via_map = rule.get("via", {})   # NEW

        ratios = TURN_SPLIT_RATIOS.get(det_key)
        split_counts = split_count(total, len(to_edges), ratios)

        for to_edge, cnt in zip(to_edges, split_counts):
            turn_rows.append({
                "intersection": det_key[0],
                "detector_id": det_key[1],
                "from_edge": from_edge,
                "to_edge": to_edge,
                "via": via_map.get(to_edge, ""),   # NEW
                "count": int(cnt)
            })

    turn_out_df = pd.DataFrame(turn_rows)

    if turn_out_df.empty:
        print("\nNo turn rows generated.")
        continue

    turn_agg = (
        turn_out_df.groupby(["from_edge", "to_edge", "via"], as_index=False)["count"]
        .sum()
        .sort_values(["from_edge", "to_edge", "via"])
    )

    print("\nTURN COUNTS")
    print(turn_agg.to_string(index=False))

    turn_root = ET.Element("data")
    turn_interval = ET.SubElement(turn_root, "interval", {
        "id": sc["name"],
        "begin": str(sc["begin"]),
        "end": str(sc["end"])
    })

    for _, r in turn_agg.iterrows():
        attrs = {
            "from": str(r["from_edge"]),
            "to": str(r["to_edge"]),
            "count": str(int(r["count"]))
        }

        if str(r["via"]).strip():
            attrs["via"] = str(r["via"]).strip()

        ET.SubElement(turn_interval, "edgeRelation", attrs)

    turn_xml_path = OUT_DIR / f"turnData_{sc['name']}.xml"
    turn_csv_path = OUT_DIR / f"turnData_{sc['name']}.csv"
    turn_debug_path = OUT_DIR / f"turnData_debug_{sc['name']}.csv"

    turn_agg.to_csv(turn_csv_path, index=False, encoding="utf-8-sig")
    turn_out_df.to_csv(turn_debug_path, index=False, encoding="utf-8-sig")

    with open(turn_xml_path, "w", encoding="utf-8") as f:
        f.write(prettify(turn_root))

    print("Saved:", turn_csv_path)
    print("Saved:", turn_debug_path)
    print("Saved:", turn_xml_path)

print("\nDONE ✅")


SCENARIO: weekday_morning

EDGE COUNTS
          edge  count
   -1182936853    152
   -1183449794     74
   -13831317#2    185
  -202157700#5    502
    -202157703    438
    -202234344    804
    -237921040   1238
   -25563346#2     44
-26202222#0.41    360
   202157700#3    418
  237921036.41    461
    26202222#0    290
      75642281    392
Saved: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\edgeData_weekday_morning.csv
Saved: C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\edgeData_weekday_morning.xml

TURN COUNTS
     from_edge      to_edge                    via  count
   -1182936853   12528053#0                           117
   -1182936853   26202222#0                            35
-13831317#2.34  -84620353#2                            82
-13831317#2.34     14811377                            40
-13831317#2.34 

** BEST RouteSampler**

& "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
  "C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
  -r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\Candidate_Routes_Trips\candidate_merged_plus_manual_plus_filtered_random.rou.xml" `
  --edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\edgeData_weekday_morning.xml" `
  --turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\turnData_weekday_morning.xml" `
  --optimize full `
  --min-count 2 `
  --minimize-vehicles 1 `
  -o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\weekday_morning_sampled_merged.rou.xml"


  **************

  & "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
  "C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
  -r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\Candidate_Routes_Trips\candidate_merged_plus_manual_plus_filtered_random.rou.xml" `
  --edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\edgeData_weekday_morning.xml" `
  --turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\turnData_weekday_morning.xml" `
  --optimize full `
  --min-count 2 `
  --minimize-vehicles 1 `
  --attributes "departLane='best' departSpeed='max' departPos='random' type='trafficMix'" `
  -o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\weekday_morning_sampled_merged.rou.xml"

  ******************************

  en son bunu aldim 
  
  & "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe" `
  "C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py" `
  -r "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\Candidate_Routes_Trips\candidate_merged_plus_manual_plus_filtered_random.rou.xml" `
  --edgedata-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\edgeData_weekday_morning.xml" `
  --turn-files "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\turnData_weekday_morning.xml" `
  --optimize full `
  --min-count 2 `
  --minimize-vehicles 1 `
  --write-route-ids `
  --attributes "departLane='best_prob' departSpeed='desired' departPos='random' type='trafficMix'" `
  -o "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau\routesampler_input\weekday_morning_sampled_merged.rou.xml"

**For all scenarios**

$pythonExe = "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe"
$routeSampler = "C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py"

$baseInput = "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau"
$candidateRoutes = "$baseInput\Candidate_Routes_Trips\candidate_merged_plus_manual_plus_filtered_random.rou.xml"
$routeSamplerInput = "$baseInput\routesampler_input"

$scenarios = @(
    "weekday_morning",
    "weekday_evening",
    "weekend_morning",
    "weekend_evening",
    "holiday_morning",
    "holiday_evening"
)

foreach ($scenario in $scenarios) {
    $edgeFile = "$routeSamplerInput\edgeData_$scenario.xml"
    $turnFile = "$routeSamplerInput\turnData_$scenario.xml"
    $outFile  = "$routeSamplerInput\${scenario}_sampled_merged.rou.xml"

    if (-not (Test-Path $edgeFile)) {
        Write-Host "SKIP: edge file missing for $scenario -> $edgeFile"
        continue
    }

    if (-not (Test-Path $turnFile)) {
        Write-Host "SKIP: turn file missing for $scenario -> $turnFile"
        continue
    }

    Write-Host ""
    Write-Host "============================================================"
    Write-Host "Running RouteSampler for: $scenario"
    Write-Host "============================================================"

    & $pythonExe `
      $routeSampler `
      -r $candidateRoutes `
      --edgedata-files $edgeFile `
      --turn-files $turnFile `
      --optimize full `
      --min-count 2 `
      --minimize-vehicles 1 `
      -o $outFile

    Write-Host "Finished: $scenario"
    Write-Host "Output : $outFile"
}

$pythonExe = "C:\Users\mogul\AppData\Local\Programs\Python\Python39\python.exe"
$routeSampler = "C:\Program Files (x86)\Eclipse\Sumo\tools\routeSampler.py"

$baseInput = "C:\Users\mogul\OneDrive\Masaüstü\mt_emre\chapter 4 - build digital twin\LSTM - detector prediction\input\landau"
$candidateRoutes = "$baseInput\Candidate_Routes_Trips\candidate_merged_plus_manual_plus_filtered_random.rou.xml"
$routeSamplerInput = "$baseInput\routesampler_input"

$scenarios = @(
    "weekday_morning",
    "weekday_evening",
    "weekend_morning",
    "weekend_evening",
    "holiday_morning",
    "holiday_evening"
)

foreach ($scenario in $scenarios) {
    $edgeFile = "$routeSamplerInput\edgeData_$scenario.xml"
    $turnFile = "$routeSamplerInput\turnData_$scenario.xml"
    $outFile  = "$routeSamplerInput\${scenario}_sampled_merged.rou.xml"

    if (-not (Test-Path $edgeFile)) {
        Write-Host "SKIP: edge file missing for $scenario -> $edgeFile"
        continue
    }

    if (-not (Test-Path $turnFile)) {
        Write-Host "SKIP: turn file missing for $scenario -> $turnFile"
        continue
    }

    Write-Host ""
    Write-Host "============================================================"
    Write-Host "Running RouteSampler for: $scenario"
    Write-Host "============================================================"

    & $pythonExe `
      $routeSampler `
      -r $candidateRoutes `
      --edgedata-files $edgeFile `
      --turn-files $turnFile `
      --optimize full `
      --min-count 2 `
      --minimize-vehicles 1 `
      --attributes "departLane='best' departSpeed='max' departPos='random'" `
      -o $outFile

    Write-Host "Finished: $scenario"
    Write-Host "Output : $outFile"
}